<a href="https://colab.research.google.com/github/lcribeiro1976/Intelig-nciaArtificial/blob/main/trabalho_final/classificacao_digitos_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalação e importação das bibliotecas

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from sklearn.metrics import classification_report, confusion_matrix

print(f' - TensorFlow versão: {tf.__version__}')
print(f' - NumPy versão: {np.__version__}')
print(' - Todas as bibliotecas carregadas com sucesso!')

In [ ]:
# Gerar e salvar o dataset como CSV
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist

(X_treino, y_treino), (X_teste, y_teste) = mnist.load_data()

colunas = [f'pixel_{i}' for i in range(784)]

df_treino = pd.DataFrame(X_treino.reshape(-1, 784), columns=colunas)
df_treino.insert(0, 'label', y_treino)
df_treino.to_csv('mnist_treino.csv', index=False)

df_teste = pd.DataFrame(X_teste.reshape(-1, 784), columns=colunas)
df_teste.insert(0, 'label', y_teste)
df_teste.to_csv('mnist_teste.csv', index=False)

print('CSVs gerados!')

In [ ]:
# Carregando o dataset MNIST diretamente do Keras

(X_treino, y_treino), (X_teste, y_teste) = mnist.load_data()

print(' - Informações do Dataset:')
print(f'   Amostras de treino : {X_treino.shape[0]}')
print(f'   Amostras de teste  : {X_teste.shape[0]}')
print(f'   Tamanho das imagens: {X_treino.shape[1]}x{X_treino.shape[2]} pixels')
print(f'   Classes (dígitos)  : {np.unique(y_treino)}')

In [ ]:
# Visualizando exemplos de imagens do dataset
plt.figure(figsize=(12, 4))
plt.suptitle('Exemplos de Dígitos Manuscritos do Dataset MNIST', fontsize=14, fontweight='bold')

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_treino[i], cmap='gray')
    plt.title(f'Dígito: {y_treino[i]}', fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Verificando a distribuição das classes
plt.figure(figsize=(10, 4))
unique, counts = np.unique(y_treino, return_counts=True)
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.title('Distribuição das Classes no Conjunto de Treino', fontsize=13)
plt.xlabel('Dígito')
plt.ylabel('Quantidade de Amostras')
plt.xticks(range(10))
plt.tight_layout()
plt.show()
print(' - O dataset é balanceado — cada dígito tem aproximadamente 6.000 amostras.')

In [ ]:
# Pré-processamento de dados

# 1. Normalização: os valores dos pixels vão de 0 a 255.
#    Dividimos por 255 para que fiquem entre 0 e 1.
#    Isso ajuda a rede a aprender mais rápido e de forma mais estável.
X_treino = X_treino.astype('float32') / 255.0
X_teste  = X_teste.astype('float32')  / 255.0

# 2. Redimensionamento: as imagens são 28x28 pixels (matriz 2D).
#    A camada Dense (totalmente conectada) espera vetores 1D.
#    Portanto, "achatamos" cada imagem em um vetor de 784 valores (28x28=784).
X_treino_flat = X_treino.reshape(-1, 784)
X_teste_flat  = X_teste.reshape(-1, 784)

print(' - Pré-processamento concluído!')
print(f'   Formato original da imagem : {X_treino.shape[1:]}')
print(f'   Formato após achatamento   : {X_treino_flat.shape[1:]}')
print(f'   Valores dos pixels: entre {X_treino_flat.min():.1f} e {X_treino_flat.max():.1f}')

In [ ]:
# Construção da rede neural

# Definindo a arquitetura da Rede Neural
# -----------------------------------------------
# Camada de Entrada : 784 neurônios (um por pixel)
# Camada Oculta 1   : 256 neurônios com ativação ReLU
# Dropout 1         : Desativa 30% dos neurônios aleatoriamente (evita overfitting)
# Camada Oculta 2   : 128 neurônios com ativação ReLU
# Dropout 2         : Desativa 20% dos neurônios aleatoriamente
# Camada de Saída   : 10 neurônios com ativação Softmax (um por classe/dígito)

modelo = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
], name='MLP_MNIST')

# Compilando o modelo
modelo.compile(
    optimizer='adam',             # Algoritmo de otimização Adam (adaptativo)
    loss='sparse_categorical_crossentropy',  # Função de perda para classificação multiclasse
    metrics=['accuracy']          # Métrica de avaliação: acurácia
)

# Resumo da arquitetura
modelo.summary()

In [ ]:
# Treinamento da rede neural

# Callback Early Stopping: interrompe o treinamento se a val_loss
# não melhorar por 5 épocas consecutivas, evitando overfitting
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print(' - Iniciando treinamento...')
historico = modelo.fit(
    X_treino_flat, y_treino,
    epochs=30,                      # Máximo de 30 épocas
    batch_size=128,                 # Processa 128 amostras por vez
    validation_split=0.1,           # 10% do treino para validação
    callbacks=[early_stopping],
    verbose=1
)
print(' - Treinamento concluído!')

In [ ]:
# Visualização do aprendizado

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Aprendizado da Rede Neural', fontsize=14, fontweight='bold')

# Gráfico de Acurácia
ax1.plot(historico.history['accuracy'],     label='Acurácia (Treino)', color='steelblue', linewidth=2)
ax1.plot(historico.history['val_accuracy'], label='Acurácia (Validação)', color='darkorange', linewidth=2, linestyle='--')
ax1.set_title('Acurácia por Época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Acurácia')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico de Perda
ax2.plot(historico.history['loss'],     label='Perda (Treino)', color='steelblue', linewidth=2)
ax2.plot(historico.history['val_loss'], label='Perda (Validação)', color='darkorange', linewidth=2, linestyle='--')
ax2.set_title('Perda por Época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Avaliação no conjunto de testes

# Avaliando o modelo nos dados de teste (nunca vistos durante o treino)
perda_teste, acuracia_teste = modelo.evaluate(X_teste_flat, y_teste, verbose=0)
print(' - Resultado Final no Conjunto de Teste:')
print(f'   Acurácia : {acuracia_teste * 100:.2f}%')
print(f'   Perda    : {perda_teste:.4f}')

In [ ]:
# Gerando as predições para o conjunto de teste
y_pred_proba = modelo.predict(X_teste_flat, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)  # Pega o índice da maior probabilidade

# Relatório detalhado por classe
print(' - Relatório de Classificação por Dígito:')
print(classification_report(y_teste, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_teste, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matriz de Confusão – Conjunto de Teste', fontsize=14, fontweight='bold')
plt.xlabel('Predição do Modelo', fontsize=12)
plt.ylabel('Valor Real', fontsize=12)
plt.tight_layout()
plt.show()

print(' - Cada linha representa o dígito real; cada coluna, o dígito predito.')
print('   Os valores na diagonal principal são os acertos.')

In [ ]:
# Visualizando 15 exemplos do conjunto de teste com as predições
plt.figure(figsize=(15, 6))
plt.suptitle('Predições do Modelo – Exemplos do Conjunto de Teste', fontsize=13, fontweight='bold')

indices = np.random.choice(len(X_teste), 15, replace=False)

for i, idx in enumerate(indices):
    plt.subplot(3, 5, i + 1)
    plt.imshow(X_teste[idx], cmap='gray')

    pred  = y_pred[idx]
    real  = y_teste[idx]
    conf  = y_pred_proba[idx][pred] * 100
    cor   = 'green' if pred == real else 'red'

    plt.title(f'Real: {real} | Pred: {pred}\n{conf:.1f}%', color=cor, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()
print('Verde = acerto | Vermelho = erro')

In [ ]:
# Exemplos onde o modelo ERROU
erros = np.where(y_pred != y_teste)[0]
print(f' - Total de erros no conjunto de teste: {len(erros)} de {len(y_teste)} amostras')

plt.figure(figsize=(12, 5))
plt.suptitle('Exemplos de Erros do Modelo', fontsize=13, fontweight='bold', color='red')

for i, idx in enumerate(erros[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_teste[idx], cmap='gray')
    plt.title(f'Real: {y_teste[idx]}\nPred: {y_pred[idx]}', color='red', fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Salvando o modelo treinado

# Salva o modelo no formato nativo do Keras
modelo.save('modelo_mnist.keras')
print(' - Modelo salvo como: modelo_mnist.keras')

# Para carregar o modelo futuramente:
# modelo_carregado = keras.models.load_model('modelo_mnist.keras')

---
## - Etapa 10 – Conclusão

Neste projeto foi desenvolvida uma **Rede Neural Artificial do tipo MLP (Multilayer Perceptron)** para classificar dígitos manuscritos do dataset MNIST.

### Resumo dos Resultados

| Métrica | Valor |
|---|---|
| Acurácia no Teste | ~98% |
| Total de Parâmetros | ~236.554 |
| Dataset | MNIST (70.000 imagens) |

### Conceitos de IA Aplicados

- **Rede Neural Feedforward (MLP)**: múltiplas camadas de neurônios artificiais conectados
- **Função de ativação ReLU**: introduz não-linearidade na aprendizagem
- **Função de ativação Softmax**: converte saídas em probabilidades por classe
- **Dropout**: técnica de regularização para evitar overfitting
- **Otimizador Adam**: algoritmo adaptativo de gradiente descendente
- **Early Stopping**: interrupção antecipada baseada na perda de validação
- **Normalização**: padronização dos valores de entrada para facilitar o aprendizado
